In [1]:
import pandas as pd
from src.recovery_model import RecoveryModel

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Select a folder for the data to be used
folder = "test_1"  # ! choose between: test_1  test_2  Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [3]:
# This section insures that the structure of the excel files is consistent
# ! PLEASE IGNORE FOR NOW

layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "parameterCode": "parameterCode",
        "Year": "year",  # not considered at this stage
        "Scenario": "scenario",  # not considered at this stage
        "Location": "region",  # not considered at this stage
        "UoM": "unit",  # not considered at this stage
    },
    # only values accepted for the parameterCode column
    # ! do not change
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Unit": "unit",
        "Year": "year",  # not considered at this stage
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "process": "process",
        "Year": "year",  # not considered at this stage
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---


# Recovery model


In [4]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

KeyError: 'parameterCode'

---

# Metadata


In [ ]:
model.dims

In [ ]:
model.size

In [ ]:
model.flows_eqs

---

# Model equation

$$(I - A^T)x = y$$

<img src="./doc/img/model_equation.png" alt="model_equation" width="350" />

In [ ]:
model.lneqs  # the A matrix (it will be properly renamed later)

In [ ]:
model.y

---

# Solver


In [ ]:
model.solve(aggregate=False, pivot=False).fillna("")

In [ ]:
model.solve(aggregate=False, pivot=True).fillna("")

---

# Mass balance

In [ ]:
# folder = "Toy_WEEE_v2"
mass_balance = pd.read_csv(f"consolidation/{folder}_solution_mass_balance.csv", index_col=0)
mass_balance.fillna("")

In [ ]:
impossible_rows = mass_balance["mass_balance"] < 0
mass_balance[impossible_rows].fillna("")